# Calculate Depression Depths (mm) per Subbasin

In [1]:
import os
import sys
import fileinput
import numpy as np
import geopandas as gpd
import pandas as pd
from shapely.geometry import Point, LineString, Polygon, MultiPoint

sys.path.append("..")
from src.py.proj_conf import *

## Import input files

In [3]:
# CLRH basin delineation shapefile
mc_clrh_gdf = gpd.read_file(DATA_RAW + "CLRH_RoutingProduct/05OE006/finalcat_info_v1-0.shp")

# CanVec hydro shapefile
mc_streams = gpd.read_file(DATA_RAW + "CanVec_Hydro/canvec_50K_MB_Hydro/water_linear_flow_1_2.shp")

## Import WhiteboxTools

In [5]:
from WBT.whitebox_tools import WhiteboxTools

wbt = WhiteboxTools()

In [18]:
wbt.set_working_dir(os.getcwd() + '/' + DATA_RAW)

In [16]:
wbt.get_working_dir()

'c:\\Users\\emilito\\Documents\\PWA\\_notebooks\\Data\\Raw\\manning'

In [21]:
os.getcwd()

'c:\\Users\\emilito\\Documents\\PWA\\_notebooks\\WBT'

In [25]:
os.path.dirname(os.path.abspath("__file__")) + "/Data/Raw/manning/"

'c:\\Users\\emilito\\Documents\\PWA\\_notebooks\\WBT/Data/Raw/manning/'

In [41]:
wbt.set_working_dir(os.path.abspath(os.path.join(os.path.dirname("__file__"), '..', '..', 'Data/Raw/manning/WB_Data')))

In [42]:
wbt.get_working_dir()

'c:\\Users\\emilito\\Documents\\PWA\\Data\\Raw\\manning\\WB_Data'

## Process DEM and Stream Network

In [50]:
# Clip NHN streams to study area
wbt.clip(
    i = 'NHN_05OE000_5_0_HD_SLWATER_1.shp',
    clip = 'finalcat_info_v1-0_Project.shp',
    output = 'NHN_Clip_wbt.shp'
)

.\whitebox_tools.exe --run="Clip" --wd="c:\Users\emilito\Documents\PWA\Data\Raw\manning\WB_Data" --input='NHN_05OE000_5_0_HD_SLWATER_1.shp' --clip='finalcat_info_v1-0_Project.shp' --output='NHN_Clip_wbt.shp' -v --compress_rasters=False

****************************
* Welcome to Clip          *
* Powered by WhiteboxTools *
* www.whiteboxgeo.com      *
****************************
Reading data...
Progress: 0%
Progress: 1%
Progress: 2%
Progress: 3%
Progress: 4%
Progress: 5%
Progress: 6%
Progress: 7%
Progress: 8%
Progress: 9%
Progress: 10%
Progress: 11%
Progress: 12%
Progress: 13%
Progress: 14%
Progress: 15%
Progress: 16%
Progress: 17%
Progress: 18%
Progress: 19%
Progress: 20%
Progress: 21%
Progress: 22%
Progress: 23%
Progress: 24%
Progress: 25%
Progress: 26%
Progress: 27%
Progress: 28%
Progress: 29%
Progress: 30%
Progress: 31%
Progress: 32%
Progress: 33%
Progress: 34%
Progress: 35%
Progress: 36%
Progress: 37%
Progress: 38%
Progress: 39%
Progress: 40%
Progress: 41%
Progress: 42%
Progress: 

0

In [48]:
# Clip dem raster to study area
wbt.clip_raster_to_polygon(
    i = 'sr_dem_cgvd28.tif',
    polygons = 'finalcat_info_v1-0.shp',
    output = 'sr_dem_cgvd28_Clip_wbt.tif'
)

.\whitebox_tools.exe --run="ClipRasterToPolygon" --wd="c:\Users\emilito\Documents\PWA\Data\Raw\manning\WB_Data" --input='sr_dem_cgvd28.tif' --polygons='finalcat_info_v1-0.shp' --output='sr_dem_cgvd28_Clip_wbt.tif' -v --compress_rasters=False

**********************************
* Welcome to ClipRasterToPolygon *
* Powered by WhiteboxTools       *
* www.whiteboxgeo.com            *
**********************************
Reading data...
memory allocation of 40703266800 bytes failed


0

In [35]:
# Resample DEM to 5m (for faster processing)
wbt.resample(
    inputs = 'sr_dem_cgvd28.tif',
    output = 'sr_dem_cgvd28_5m.tif',
    cell_size = 5,
    method = "cc"
)

.\whitebox_tools.exe --run="Resample" --wd="c:\Users\emilito\Documents\PWA\Data\Raw\manning\DEMs" --inputs='sr_dem_cgvd28.tif' --output='sr_dem_cgvd28_5m.tif' --cell_size='5' --method=cc -v --compress_rasters=False

****************************
* Welcome to Resample      *
* Powered by WhiteboxTools *
* www.whiteboxgeo.com      *
****************************
Reading data...
memory allocation of 40703266800 bytes failed


0

In [57]:
# Convert stream network to raster
wbt.vector_lines_to_raster(
    i = 'NHN_Clip_wbt_Project.shp',
    output = 'NHN_Clip_Project_raster_wbt.tif',
    field = "FID",
    nodata = True,
    cell_size = 5.0
)

.\whitebox_tools.exe --run="VectorLinesToRaster" --wd="c:\Users\emilito\Documents\PWA\Data\Raw\manning\WB_Data" --input='NHN_Clip_wbt_Project.shp' --field=FID --output='NHN_Clip_Project_raster_wbt.tif' --nodata --cell_size='5.0' -v --compress_rasters=False

**********************************
* Welcome to VectorLinesToRaster *
* Powered by WhiteboxTools       *
* www.whiteboxgeo.com            *
**********************************
Reading data...
Reading attributes: 0%
Reading attributes: 1%
Reading attributes: 2%
Reading attributes: 3%
Reading attributes: 4%
Reading attributes: 5%
Reading attributes: 6%
Reading attributes: 7%
Reading attributes: 8%
Reading attributes: 9%
Reading attributes: 10%
Reading attributes: 11%
Reading attributes: 12%
Reading attributes: 13%
Reading attributes: 14%
Reading attributes: 15%
Reading attributes: 16%
Reading attributes: 17%
Reading attributes: 18%
Reading attributes: 19%
Reading attributes: 20%
Reading attributes: 21%
Reading attributes: 22%
Reading a

0